# Hangman greedy-strategy explorer

Look at the greedy strategy's results from `strategy.py`. Pick a word length,
then explore: the best opening letters, which words are hardest, how average
misses scale with length, and how tweaking per-word frequencies moves the number.

Reminders: a *miss* is a wrong letter guess; frequencies are non-normalized and
default to 1 per word; the greedy rule guesses the un-guessed letter with the
highest combined frequency of candidate words that contain it.

In [ ]:
from pathlib import Path
import sys

def find_math_dir():
    cwd = Path.cwd().resolve()
    for c in (cwd, cwd.parent, cwd.parent.parent, cwd / 'math', cwd.parent / 'math'):
        if (c / 'strategy.py').exists():
            return c
    raise FileNotFoundError('Could not find strategy.py. Open from the repo root, math/, or math/notebooks/.')

math_dir = find_math_dir()
if str(math_dir) not in sys.path:
    sys.path.insert(0, str(math_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import strategy

print(f'imported strategy from {math_dir}')

## Pick a length

Change `LENGTH` and re-run the rest. `GreedyGame` is built once here; the
analysis cells below reuse it.

In [ ]:
LENGTH = 5          # <-- choose a word length

g = strategy.GreedyGame(LENGTH)
print(f'{g.N:,} words of length {LENGTH}  (zipf >= 2.5)')
print(f'greedy opens with: {g.greedy_first_letter()!r}')
print(f'Mode A  average misses: {g.average_misses():.4f}')

## Best opening letters (Mode B)

Expected misses if each letter is forced as the first guess, then greedy is
played. The dashed line is the greedy opener (the best one).

In [ ]:
by_letter = g.misses_by_starting_letter()          # sorted best-first
mb = pd.Series(by_letter, name='avg_misses')
mb.index.name = 'first_letter'

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(mb.index, mb.values, color='#3b6ea5')
ax.axhline(g.average_misses(), color='#cc4444', ls='--', lw=1, label='greedy opener')
ax.set_ylabel('avg misses')
ax.set_title(f'Expected misses by opening letter (length {LENGTH})')
ax.legend()
plt.show()

mb.to_frame().T

## Per-word difficulty

How many misses the greedy strategy makes on each word, and the distribution.

In [ ]:
pwm = g.per_word_misses()
words = pd.DataFrame({'word': g.words, 'misses': pwm.astype(int)})

print(f'solved with 0 misses: {(pwm == 0).sum():,} words')
print(f'max misses: {int(pwm.max())}   mean: {pwm.mean():.3f}')

fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(pwm, bins=range(0, int(pwm.max()) + 2), color='#3b6ea5', align='left', rwidth=0.85)
ax.set_xlabel('misses'); ax.set_ylabel('# words')
ax.set_title(f'Misses per word (length {LENGTH})')
plt.show()

print('hardest words:')
words.sort_values('misses', ascending=False).head(12).reset_index(drop=True)

## Trace one word

The actual greedy guess sequence for a word: each guess and whether it hit.

In [ ]:
WORD = g.words[int(pwm.argmax())]      # the hardest word; or set your own, e.g. 'bench'

hits = g.trace(WORD)
n_miss = sum(1 for _, hit in hits if not hit)
print(f'{WORD!r}: {n_miss} misses')
for letter, hit in hits:
    mark = 'hit ' if hit else 'MISS'
    print(f'  {letter}  {mark}')

## Average misses by length

How the greedy average scales across word lengths (builds a `GreedyGame` per
length; Mode A is fast).

In [ ]:
lengths = list(range(3, 13))
avgs = [strategy.GreedyGame(L).average_misses() for L in lengths]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(lengths, avgs, 'o-', color='#3b6ea5')
ax.set_xlabel('word length'); ax.set_ylabel('avg misses (greedy)')
ax.set_title('Greedy average misses by word length')
ax.grid(alpha=0.3)
plt.show()

pd.DataFrame({'length': lengths, 'avg_misses': [round(a, 3) for a in avgs]})

## Tweaking frequencies

Frequencies default to 1 for every word. Build the `GreedyGame` once, then pass
a `freq` vector to re-solve (~25 ms each). `freq` can be a numpy array aligned to
`g.words`, a `{word: frequency}` dict, or a `callable(word) -> float`.

In [ ]:
w0 = g.words[0]        # the most frequent word of this length (list is freq-sorted)

f = g.ones()
f[0] = 100.0           # make w0 dominate the weighting
print(f'boosting {w0!r} 100x:')
print(f'  baseline avg : {g.average_misses():.4f}')
print(f'  tweaked  avg : {g.average_misses(f):.4f}')

# dict form (missing words default to 1):
# g.average_misses({w0: 100.0})